# [SOLUTION] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

### Vocareum OpenAI Proxy
In the Vocareum cloud workspace, OpenAI API requests should be routed through the Vocareum proxy.
Ensure `OPENAI_API_BASE` is set in your `.env` file (e.g., `https://openai.vocareum.com/v1`).

In [4]:
# Load environment variables
load_dotenv('.env', override=True)

# Configure OpenAI to use the Vocareum proxy URL if provided
import openai
openai.base_url = os.getenv("OPENAI_API_BASE", openai.base_url)

### VectorDB Instance

In [5]:
# Instantiate ChromaDB Client
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
# OpenAI embedding function
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv('CHROMA_OPENAI_API_KEY'),
    api_base=os.getenv('OPENAI_API_BASE')
)

In [7]:
# Get or create collection (avoids error if collection already exists)
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [8]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

files = sorted(os.listdir(data_dir))
json_files = [f for f in files if f.endswith(".json")]
print(f"Found {len(json_files)} JSON files to process")

for i, file_name in enumerate(json_files):
    print(f"Processing file {i+1}/{len(json_files)}: {file_name}")
    
    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    # Check if document already exists to avoid duplicate ID error
    try:
        existing = collection.get(ids=[doc_id])
        if existing['ids']:
            print(f"Document {doc_id} already exists, skipping...")
            continue
    except Exception as e:
        print(f"Error checking existing document {doc_id}: {e}")
        pass  # If get fails, proceed to add

    try:
        collection.add(
            ids=[doc_id],
            documents=[content],
            metadatas=[game]
        )
        print(f"Successfully added document {doc_id}")
    except Exception as e:
        print(f"Error adding document {doc_id}: {e}")
        break  # Stop on first error to debug

print("Document processing complete")

Found 15 JSON files to process
Processing file 1/15: 001.json
Document 001 already exists, skipping...
Processing file 2/15: 002.json
Document 002 already exists, skipping...
Processing file 3/15: 003.json
Document 003 already exists, skipping...
Processing file 4/15: 004.json
Document 004 already exists, skipping...
Processing file 5/15: 005.json
Document 005 already exists, skipping...
Processing file 6/15: 006.json
Document 006 already exists, skipping...
Processing file 7/15: 007.json
Document 007 already exists, skipping...
Processing file 8/15: 008.json
Document 008 already exists, skipping...
Processing file 9/15: 009.json
Document 009 already exists, skipping...
Processing file 10/15: 010.json
Document 010 already exists, skipping...
Processing file 11/15: 011.json
Document 011 already exists, skipping...
Processing file 12/15: 012.json
Document 012 already exists, skipping...
Processing file 13/15: 013.json
Document 013 already exists, skipping...
Processing file 14/15: 014.js

### Test the Collection

In [9]:
# Test semantic search
query = "What racing games are available?"
results = collection.query(
    query_texts=[query],
    n_results=3
)

print("Query:", query)
print("Results:")
for i, doc in enumerate(results['documents'][0]):
    print(f"{i+1}. {doc}")
    print(f"   Metadata: {results['metadatas'][0][i]}")
    print()

Query: What racing games are available?
Results:
1. [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.
   Metadata: {'Publisher': 'Sony Computer Entertainment', 'Genre': 'Racing', 'Name': 'Gran Turismo 5', 'Platform': 'PlayStation 3', 'Description': 'A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.', 'YearOfRelease': 2010}

2. [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.
   Metadata: {'Platform': 'PlayStation 1', 'YearOfRelease': 1997, 'Name': 'Gran Turismo', 'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.', 'Genre': 'Racing', 'Publisher': 'Sony Computer Entertainment'}

3. [Nintendo Switch] Mario Kart 8 Deluxe (2017) - An enhanced version